# Fase 3 — Validacion y leakage

Objetivo (spec, seccion 8): identificar las unidades de dependencia reales
del dataset, comparar como minimo V0 (StratifiedKFold aleatorio), V1
(group-aware) y V2 (holdout temporal), cuantificar cuanto infla el ROC-AUC
una validacion aleatoria ingenua (H1), congelar el holdout final, y aplicar
el checklist de 5 preguntas de `.claude/rules/leakage-and-validation.md` a
las columnas sospechosas de leakage.

Ver tambien: `artifacts/reports/eda_report.md` (hipotesis 1-3, 6),
`artifacts/reports/leakage_checklist_fase3.md` (checklist detallado),
`README.md` (seccion Validation Strategy con la decision final).

## 1. Unidades de dependencia reales

`Driver` no es un identificador de piloto confiable (887 valores unicos,
no se comporta como grid real de F1 — hallazgo de Fase 1/2). `Race` sola
tampoco identifica un evento unico: las mismas 26 carreras se repiten en
los 4 anios del dataset (train y test comparten las 26 carreras y los 4
anios). La clave de agrupacion valida para "evento de carrera" es
`(Race, Year)`.

In [ ]:
import pandas as pd

from f1pitstop.data.ingest import load_raw
from f1pitstop.data.split import make_group_key

train, test, sample_sub, reports = load_raw(data_dir="../data/raw")
groups = make_group_key(train)

print(f"filas train: {len(train)}")
print(f"grupos (Race, Year) unicos en train: {groups.nunique()}")
print(f"carreras (Race) unicas: {train['Race'].nunique()}")
print(f"anios (Year) unicos: {sorted(train['Year'].unique())}")

filas train: 439140
grupos (Race, Year) unicos en train: 104
carreras (Race) unicas: 26
anios (Year) unicos: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 2. Hallazgo adicional (Fase 3): anomalia de tasa de pit en 2023

Antes de disenar V2, se descubrio que el anio 2023 tiene una tasa de pit
(`PitNextLap` y `PitStop`) anormalmente baja en **todas** las carreras de
ese anio (~1% vs ~19-30% en 2022/2024/2025), no solo en `Pre-Season
Testing`. Esto es un hallazgo nuevo de Fase 3, no reportado en el EDA de
Fase 2 (que no desagrego la tasa de pit por `Year`).

In [ ]:
print("Tasa de PitNextLap por anio:")
print(train.groupby("Year")["PitNextLap"].mean().round(4))
print()
print("Tasa de PitStop por anio:")
print(train.groupby("Year")["PitStop"].mean().round(4))
print()
print("Ejemplo: misma carrera (Bahrain), distintos anios:")
print(train[train["Race"] == "Bahrain Grand Prix"].groupby("Year")["PitNextLap"].mean().round(3))

Tasa de PitNextLap por anio:
Year
2022    0.2665
2023    0.0096
2024    0.2953
2025    0.2844
Name: PitNextLap, dtype: float64

Tasa de PitStop por anio:
Year
2022    0.1865
2023    0.0124
2024    0.1922
2025    0.1957
Name: PitStop, dtype: float64

Ejemplo: misma carrera (Bahrain), distintos anios:
Year
2022    0.417
2023    0.014
2024    0.412
2025    0.477
Name: PitNextLap, dtype: float64


**Interpretacion:** esto es evidencia adicional (mas fuerte que la de Fase 2)
de que el dataset sintetico no es homogeneo entre anios — hay drift real
inducido por el propio proceso de generacion. Esto refuerza por que V2
(holdout temporal) es una comparacion relevante: un split que ignora `Year`
mezclaria en el mismo fold vueltas de un anio con comportamiento casi
degenerado (2023) con vueltas de anios normales, algo que no puede pasar en
produccion (una carrera nueva pertenece a un unico momento en el tiempo).

No se corrige ni se filtra el anio 2023 — se documenta y se deja tal cual
en el conjunto de desarrollo (asi se veria en produccion: el modelo no
elige que datos historicos recibe).

## 3. Holdout final (congelado, NUNCA se usa para decisiones de modelado)

Se congela con `freeze_final_holdout()`: `Year == 2025` queda completamente
fuera de cualquier decision de modelado hasta la evaluacion confirmatoria de
la Fase 13 (regla no negociable 6 de CLAUDE.md). Los ids ya estan
persistidos en `artifacts/tables/final_holdout_ids.csv` (generado antes de
este notebook, ver `HANDOFF.md`). Aqui solo se recarga para mostrar el
resultado — no se re-genera ni se evalua ningun modelo sobre el.

In [ ]:
from f1pitstop.data.split import load_frozen_holdout_ids

holdout_ids = load_frozen_holdout_ids("../artifacts/tables/final_holdout_ids.csv")
dev = train[~train["id"].isin(holdout_ids)]
holdout = train[train["id"].isin(holdout_ids)]

print(f"dev: {len(dev)} filas, {make_group_key(dev).nunique()} grupos (Race, Year)")
print(f"holdout congelado: {len(holdout)} filas, {make_group_key(holdout).nunique()} grupos (Race, Year)")
print(f"anios en holdout: {sorted(holdout['Year'].unique())}")
assert set(make_group_key(dev)) & set(make_group_key(holdout)) == set(), "no debe haber overlap de grupos"
print("OK: ningun grupo (Race, Year) se solapa entre dev y holdout.")

dev: 346246 filas, 78 grupos (Race, Year)
holdout congelado: 92894 filas, 26 grupos (Race, Year)
anios en holdout: [np.int64(2025)]
OK: ningun grupo (Race, Year) se solapa entre dev y holdout.


## 4. Cuantificar H1: V0 (aleatorio) vs V1 (group-aware) vs V2 (temporal, demo)

Todo lo siguiente corre **solo sobre `dev`** (2022-2024) — el holdout 2025
no se toca. Modelo deliberadamente simple
(`HistGradientBoostingClassifier`) sobre features crudas, excluyendo las
columnas `SUSPECTED_LEAKAGE` (se vetan aparte en la seccion 5): el objetivo
es aislar el efecto de la estrategia de split, no medir el mejor modelo
posible (eso es Fase 4+).

Reproducible desde script: `scripts/phase3_quantify_h1.py`. Resultado ya
guardado en `artifacts/tables/cv_strategy_comparison.csv`; aqui se recarga
para no duplicar el entrenamiento.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
comparison = pd.read_csv("../artifacts/tables/cv_strategy_comparison.csv")
comparison

                               strategy  roc_auc_mean  roc_auc_std  n_folds                                               note
0                       V0_random_kfold      0.844454     0.000908        5  ignora grupos (Race, Year); baseline optimista...
1                        V1_group_kfold      0.814568     0.022877        5  carrera (Race, Year) nunca aparece en train y ...
2  V2_temporal_2022_2023_train_2024_val      0.839152          NaN        1  demostracion temporal dentro de dev; no usa el...


**Lectura de la tabla:**

- **V0 (aleatorio)** ROC-AUC ≈ 0.844, con desviacion casi nula entre folds
  (std ≈ 0.001) — parece "muy estable", pero esa estabilidad es enganosa:
  es estable porque cada fold ve informacion de practicamente todas las
  carreras, incluyendo la carrera que esta evaluando.
- **V1 (group-aware por `(Race, Year)`)** ROC-AUC ≈ 0.815, con desviacion
  bastante mayor entre folds (std ≈ 0.023) — la variabilidad real entre
  carreras (algunas mas faciles/dificiles de predecir que otras) queda
  expuesta en vez de promediada.
- **Gap V0 − V1 ≈ 0.030 ROC-AUC.** Esto es la inflacion optimista de una
  validacion aleatoria ingenua frente al escenario real de produccion
  (confirma H1 del spec: *"un split que ignora al agrupador infla el
  ROC-AUC de forma optimista"*).
- **V2 (demo temporal, train 2022-2023 → val 2024)** ROC-AUC ≈ 0.839, un
  unico fold, mas cercano a V0 que a V1. Esto es parcialmente un artefacto
  de que 2023 (anomalo, ver seccion 2) esta en train de este fold
  particular y 2024 (normal) es la validacion — un solo fold temporal no
  captura la variabilidad entre carreras que si expone V1 con 5 folds, y
  no debe interpretarse como "V2 es mejor que V1".

## 5. Checklist de leakage — columnas sospechosas

Aplicado el checklist de 5 preguntas de
`.claude/rules/leakage-and-validation.md` a `PitStop`, `Position`,
`LapTime (s)` y a las columnas `SUSPECTED_LEAKAGE`
(`LapTime_Delta`, `Cumulative_Degradation`, `RaceProgress`,
`Position_Change`). Detalle completo, evidencia empirica y la decision
final (incluir/excluir de la Fase 4) en
`artifacts/reports/leakage_checklist_fase3.md`. Revisado ademas por el
subagente de solo lectura `leakage-auditor` antes de cerrar la fase (sin
hallazgos bloqueantes; dos huecos de documentacion senalados y cerrados
en la misma revision, ver el reporte).

**Resumen de la decision:**

| columna | decision | motivo corto |
|---|---|---|
| `PitStop` | **incluir** | describe el estado de la vuelta actual (t), AUC univariada moderada (0.52), sin evidencia de fuga |
| `Position` | **incluir** | valor de estado en tiempo real de la vuelta t, AUC univariada baja (0.52) |
| `LapTime (s)` | **incluir** | tiempo de la vuelta ya completada (t), conocido antes de decidir sobre t+1, AUC univariada baja (0.54) |
| `RaceProgress` | **excluir del set "leakage-safe" por ahora** | el denominador implicito (vueltas totales) NO es monotono en 24.2% de los grupos — misma inconsistencia que `Stint`; se penso inicialmente que era seguro y se corrigio tras la auditoria |
| `Cumulative_Degradation` | **excluir del set "leakage-safe" por ahora** | no se puede confirmar con certeza el instante exacto de corte (ver evidencia empirica); calculada sobre secuencia oculta subsampleada |
| `LapTime_Delta` | **excluir del set "leakage-safe" por ahora** | mismo motivo que `Cumulative_Degradation` |
| `Position_Change` | **excluir del set "leakage-safe" por ahora** | Fase 2 ya demostro que no coincide con la diferencia real entre filas visibles consecutivas |

Regla aplicada: `.claude/rules/leakage-and-validation.md` seccion 4 —
"si la respuesta a la pregunta 1 o 2 no se puede responder con certeza, la
feature se descarta o se corrige, no se deja pasar documentada como
limitacion". Estas cuatro columnas quedan disponibles para un experimento
ablation dedicado en Fase 6 (hipotesis 1 de `eda_report.md`), no se
eliminan del dataset, solo se excluyen del set "leakage-safe" por
defecto.

Feature set "leakage-safe" resultante para Fase 4: `LapNumber`,
`TyreLife`, `Stint`, `Position`, `PitStop`, `Compound`, `LapTime (s)`.

## 6. Decision final: estrategia oficial de CV

**Se elige V1 (StratifiedGroupKFold por `(Race, Year)`, 5 folds) como la
estrategia oficial de cross-validation para el resto del proyecto
(Fases 4-12).**

Justificacion:

1. **Corresponde al escenario que el proyecto afirma simular.** La pregunta
   de portafolio (`CLAUDE.md`) es sobre un pipeline "leakage-aware"; la
   pregunta de negocio real es "¿el modelo predice bien en una carrera que
   nunca vio?" — no "¿el modelo predice bien una vuelta de una carrera que
   ya vio en parte?". V0 responde la pregunta equivocada.
2. **V0 esta inflado ~0.03 ROC-AUC** de forma medible (seccion 4), y esa
   inflacion es exactamente el tipo de sesgo optimista que
   `.claude/rules/leakage-and-validation.md` pide detectar y evitar.
3. **V1 es mas informativo que V2 para uso repetido durante todo el
   proyecto**: da 5 estimaciones (con su propia desviacion, que expone la
   heterogeneidad real entre carreras) en vez de una sola. V2 se reserva
   para el holdout final (Fase 13) y como chequeo puntual adicional, no
   como la estrategia de todos los dias.
4. **No se elige la estrategia "mas dificil" por default** (regla del
   spec): se elige V1 porque coincide con el mecanismo de generalizacion
   que de verdad interesa, no porque de el numero mas bajo.

El holdout final (`Year == 2025`, congelado en
`artifacts/tables/final_holdout_ids.csv`) sigue las mismas reglas de no
overlap de grupos que V1, y se evalua una unica vez en la Fase 13
(regla no negociable 6 de CLAUDE.md).